# Tiny LLM Lab — Colab end-to-end training

Notebook này chạy toàn bộ pipeline bằng Python API, không gọi `train_architectures.py` hay CLI training. Mục tiêu là nhìn thấy từng bước: data contract → tokenizer/token artifacts → batch loader → model → loss → optimizer → training → checkpoint → evaluation → KV-cache benchmark.

Mặc định notebook train một model GQA khoảng 10–12M parameters trên TinyStories 10M tokens. Sau khi hiểu pipeline, có thể đổi `ARCHITECTURE` sang `mha`, `mla`, `moe` hoặc `v4` ở cell model.

## 0. Quy ước trước khi chạy

- Hãy chạy các cell theo thứ tự từ trên xuống.
- Dataset và checkpoint được lưu trên Google Drive để không mất khi Colab reset.
- Nếu đã có `tinystories_pilot_10m.manifest.json`, notebook sẽ dùng lại artifact đó. Nếu chưa có, cell dataset sẽ tải TinyStories streaming và tạo artifact mới.
- 10M tokens là experiment thật nhỏ để học pipeline; chưa phải training recipe production.

In [ ]:
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

# Sửa path này nếu repo của bạn nằm ở vị trí khác trên Drive.
PROJECT_ROOT = Path('/content/drive/MyDrive/tiny-llm-lab/llm-lab')
assert (PROJECT_ROOT / 'src' / 'llm').exists(), (
    f'Không tìm thấy project tại {PROJECT_ROOT}. Hãy upload/clone repo vào Drive trước.'
)
print(PROJECT_ROOT)

## 1. Cài dependency và import source code

Colab đã có PyTorch. Cell dưới chỉ cài các package data/tokenizer cần thiết; đây không phải cách chạy training CLI.

In [ ]:
%pip install -q numpy tokenizers tiktoken datasets matplotlib

import sys
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import json
from contextlib import nullcontext
import math
import random
import time
from dataclasses import asdict

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

from llm.benchmarking.compute import (
    active_parameter_count,
    collect_kv_cache_stats,
    estimate_flops,
    estimate_kv_cache,
)
from llm.benchmarking.inference import run_generation, synchronize
from llm.data import load_token_artifacts
from llm.data.datasets import NextTokenDataset, StatefulBatchLoader, take_token_budget
from llm.data.manifest import build_manifest
from llm.data.splits import split_documents_three
from llm.data.tokenizer import build_tokenizer
from llm.evaluation.loss import evaluate_loss_stats
from llm.models.registry import build_model, model_metadata, parameter_count

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__)
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Chọn experiment và seed

Mọi architecture phải dùng cùng các giá trị này nếu muốn so sánh công bằng. MoE là ngoại lệ về **total resident parameters**, nhưng active parameters/FLOPs vẫn được report riêng.

In [ ]:
SEED = 42
ARCHITECTURE = 'gqa'  # đổi thành: mha, gqa, mla, moe, v4
TRAIN_TOKENS = 10_000_000
VALIDATION_TOKENS = 500_000
TEST_TOKENS = 500_000
MAX_EXAMPLES = 60_000
CONTEXT_LENGTH = 256
BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 8
EVAL_EVERY_STEPS = 50
EVAL_BATCHES_DURING_TRAINING = 20
SAVE_EVERY_STEPS = 100

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATA_DIR = PROJECT_ROOT / 'data'
RUN_DIR = PROJECT_ROOT / 'runs' / f'colab_{ARCHITECTURE}_10m_notebook'
DATA_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Run directory:', RUN_DIR)

## 3. Tạo hoặc load fixed-token artifact

Tại sao cần bước này? Nếu mỗi model tự fit tokenizer và encode lại raw JSONL, số token thực tế có thể khác nhau. Artifact cố định giúp mọi model nhìn đúng cùng token stream.

Artifact gồm:

```text
train.npy
validation.npy
test.npy
tokenizer.json
manifest.json
```

In [ ]:
ARTIFACT_BASE = DATA_DIR / 'tinystories_colab_10m'
MANIFEST_PATH = ARTIFACT_BASE.with_suffix('.manifest.json')

if not MANIFEST_PATH.exists():
    from datasets import load_dataset

    print('Chưa có artifact — tải TinyStories streaming...')
    stream = load_dataset('roneneldan/TinyStories', split='train', streaming=True)
    documents = []
    for row in stream:
        text = str(row['text']).strip()
        if text:
            documents.append(text)
        if len(documents) >= MAX_EXAMPLES:
            break
    print('Documents:', len(documents))

    train_docs, validation_docs, test_docs = split_documents_three(
        documents, train_fraction=0.9, validation_fraction=0.05, seed=SEED
    )
    tokenizer = build_tokenizer('bpe', train_docs, vocab_size=16_000, min_frequency=2)
    train_full = tokenizer.encode_documents(train_docs)
    validation_full = tokenizer.encode_documents(validation_docs)
    test_full = tokenizer.encode_documents(test_docs)
    train_ids = np.asarray(take_token_budget(train_full, TRAIN_TOKENS), dtype=np.uint32)
    validation_ids = np.asarray(take_token_budget(validation_full, VALIDATION_TOKENS), dtype=np.uint32)
    test_ids = np.asarray(take_token_budget(test_full, TEST_TOKENS), dtype=np.uint32)

    tokenizer_path = ARTIFACT_BASE.with_suffix('.tokenizer.json')
    train_path = ARTIFACT_BASE.with_suffix('.train.npy')
    validation_path = ARTIFACT_BASE.with_suffix('.validation.npy')
    test_path = ARTIFACT_BASE.with_suffix('.test.npy')
    tokenizer.save(tokenizer_path)
    np.save(train_path, train_ids)
    np.save(validation_path, validation_ids)
    np.save(test_path, test_ids)

    import hashlib
    tokenizer_hash = hashlib.sha256(tokenizer_path.read_bytes()).hexdigest()
    source = {
        'id': 'roneneldan/TinyStories',
        'url': 'https://huggingface.co/datasets/roneneldan/TinyStories',
        'license': 'CDLA-Sharing-1.0',
        'license_url': 'https://cdla.dev/sharing-1-0/',
    }
    manifest = build_manifest(
        documents, train_docs, validation_docs, source, SEED, 0.9, test_docs,
        tokenizer_kind='byte_level_bpe',
        tokenizer_vocab_size=tokenizer.vocab_size,
        tokenizer_sha256=tokenizer_hash,
        train_token_count=len(train_ids),
        validation_token_count=len(validation_ids),
        test_token_count=len(test_ids),
        target_train_tokens=TRAIN_TOKENS,
        target_validation_tokens=VALIDATION_TOKENS,
        target_test_tokens=TEST_TOKENS,
        token_id_dtype='uint32',
    ).to_dict()
    manifest['data_file'] = str(ARTIFACT_BASE.with_suffix('.jsonl'))
    manifest['token_artifacts'] = {
        'train': str(train_path),
        'validation': str(validation_path),
        'test': str(test_path),
        'tokenizer': str(tokenizer_path),
    }
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    print('Đã tạo artifact:', MANIFEST_PATH)
else:
    print('Dùng artifact có sẵn:', MANIFEST_PATH)

In [ ]:
artifacts = load_token_artifacts(MANIFEST_PATH)
train_ids = artifacts.train_tokens
validation_ids = artifacts.validation_tokens
test_ids = artifacts.test_tokens
tokenizer = artifacts.tokenizer

print('Tokenizer vocab:', tokenizer.vocab_size)
print('Train tokens:', len(train_ids))
print('Validation tokens:', len(validation_ids))
print('Test tokens:', len(test_ids))
print('Contract:')
print(json.dumps(artifacts.contract, indent=2))

## 4. Biến token stream thành next-token batches

Với context `L`, một sample có dạng:

```text
input : t0 t1 t2 ... t(L-1)
target: t1 t2 t3 ... tL
```

`drop_last=True` cho train để mỗi optimizer update có đúng số token. Validation/test giữ batch cuối để final evaluation không bỏ token.

In [ ]:
train_dataset = NextTokenDataset(train_ids, CONTEXT_LENGTH, stride=CONTEXT_LENGTH)
validation_dataset = NextTokenDataset(validation_ids, CONTEXT_LENGTH, stride=CONTEXT_LENGTH)
test_dataset = NextTokenDataset(test_ids, CONTEXT_LENGTH, stride=CONTEXT_LENGTH)

train_loader = StatefulBatchLoader(
    train_dataset, BATCH_SIZE, shuffle=True, seed=SEED, drop_last=True
)
validation_loader = StatefulBatchLoader(
    validation_dataset, BATCH_SIZE, shuffle=False, seed=SEED, drop_last=False
)
test_loader = StatefulBatchLoader(
    test_dataset, BATCH_SIZE, shuffle=False, seed=SEED, drop_last=False
)

TOKENS_PER_UPDATE = BATCH_SIZE * CONTEXT_LENGTH * GRADIENT_ACCUMULATION_STEPS
MAX_STEPS = TRAIN_TOKENS // TOKENS_PER_UPDATE
print('Train windows:', len(train_dataset))
print('Tokens per optimizer update:', TOKENS_PER_UPDATE)
print('Optimizer steps for ~10M tokens:', MAX_STEPS)

## 5. Build model

Đây là chỗ thay architecture. Các model cùng nhận `input_ids` và trả logits `(batch, sequence, vocab)`. `use_cache=False` trong pretraining vì KV cache chỉ dành cho autoregressive inference.

In [ ]:
base_model_config = {
    'vocab_size': tokenizer.vocab_size,
    'context_length': CONTEXT_LENGTH,
    'emb_dim': 256,
    'n_heads': 8,
    'n_layers': 8,
    'hidden_dim': 1024,
    'drop_rate': 0.0,
    'dropout': 0.0,
    'qkv_bias': False,
    'tie_embeddings': True,
    'n_kv_groups': 2,
    'latent_dim': 64,
    'head_dim': 32,
    'q_lora_rank': 64,
    'rope_dim': 16,
    'rope_base': 10_000.0,
    'window_size': 64,
    'compress_ratios': [0, 2, 0, 2, 0, 2, 0, 2],
    'index_topk': 4,
    'moe_hidden_dim': 512,
    'shared_hidden_dim': 512,
    'num_experts': 2,
    'num_experts_per_tok': 1,
    'shared_expert_hidden_dim': 512,
    'norm_eps': 1e-6,
}

architecture_overrides = {
    'mla': {'hidden_dim': 896, 'num_experts': 0, 'num_experts_per_tok': 0},
    'moe': {'hidden_dim': 768, 'num_experts': 4, 'num_experts_per_tok': 1, 'shared_expert_hidden_dim': 256},
    'v4': {'n_layers': 5, 'compress_ratios': [0, 2, 0, 2, 0], 'q_lora_rank': 32, 'latent_dim': 32, 'moe_hidden_dim': 256, 'shared_hidden_dim': 256},
}
model_config = {**base_model_config, **architecture_overrides.get(ARCHITECTURE, {})}
model = build_model(ARCHITECTURE, model_config).to(device)
metadata = model_metadata(ARCHITECTURE, model)
flops = estimate_flops(model, ARCHITECTURE)
kv = estimate_kv_cache(model, ARCHITECTURE, CONTEXT_LENGTH)
print(json.dumps({
    'architecture': ARCHITECTURE,
    'total_parameters': parameter_count(model),
    'active_parameters': active_parameter_count(model, ARCHITECTURE),
    'forward_flops_per_token': flops.forward_flops_per_token,
    'training_flops_per_token': flops.training_flops_per_token,
    'kv_cache_bytes_per_token': kv.bytes_per_token,
}, indent=2))

In [ ]:
# Sanity check: một forward pass trước khi train.
inputs, targets = next(iter(train_loader))
inputs = inputs.to(device)
targets = targets.to(device)
with torch.no_grad():
    logits = model(inputs, use_cache=False)
    initial_loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
print('inputs:', tuple(inputs.shape))
print('logits:', tuple(logits.shape))
print('initial loss:', float(initial_loss))

## 6. Optimizer, scheduler và mixed precision

Mỗi microbatch tạo gradient. Sau `GRADIENT_ACCUMULATION_STEPS` microbatch mới gọi `optimizer.step()`. Vì vậy effective token budget/update lớn hơn batch vật lý.

In [ ]:
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
WARMUP_STEPS = max(1, min(60, MAX_STEPS // 10))
MIN_LR_RATIO = 0.1
GRAD_CLIP_NORM = 1.0

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    betas=(0.9, 0.95),
    weight_decay=WEIGHT_DECAY,
)

def lr_lambda(step):
    if step <= WARMUP_STEPS:
        return step / WARMUP_STEPS
    progress = min(1.0, (step - WARMUP_STEPS) / max(1, MAX_STEPS - WARMUP_STEPS))
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return MIN_LR_RATIO + (1.0 - MIN_LR_RATIO) * cosine

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
use_amp = device.type == 'cuda'
amp_dtype = torch.bfloat16 if use_amp and torch.cuda.is_bf16_supported() else torch.float16
use_scaler = use_amp and amp_dtype == torch.float16
scaler = torch.amp.GradScaler('cuda', enabled=use_scaler)
print('AMP:', use_amp, amp_dtype if use_amp else 'disabled')

## 7. Evaluation và checkpoint helpers

Training evaluation chỉ lấy một số batch để monitor nhanh. Final validation/test bên dưới sẽ chạy toàn bộ held-out stream. Loss được tính bằng tổng negative log-likelihood chia cho tổng target tokens, không phải trung bình ngây thơ của batch losses.

In [ ]:
history = []
tokens_seen = 0
best_validation_loss = float('inf')
train_iterator = iter(train_loader)
start_time = time.perf_counter()

def evaluate_loader(loader, max_batches=None):
    return evaluate_loss_stats(model, loader, device, max_batches)

def save_notebook_checkpoint(path, step):
    checkpoint = {
        'architecture': ARCHITECTURE,
        'model_config': model_config,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'scaler_state': scaler.state_dict() if use_scaler else None,
        'step': step,
        'tokens_seen': tokens_seen,
        'history': history,
        'data_manifest': artifacts.manifest,
        'data_contract': artifacts.contract,
        'tokenizer': tokenizer.to_state(),
        'flops': flops.as_dict(),
        'kv_cache': kv.as_dict(),
    }
    torch.save(checkpoint, path)
    print('saved:', path)

## 8. Training loop — cell quan trọng nhất

Đọc chậm từng phần:

1. Lấy `GRADIENT_ACCUMULATION_STEPS` microbatches.
2. Forward với `use_cache=False`.
3. Tính causal cross-entropy.
4. Chia loss cho số microbatch trước backward.
5. Clip gradient, update optimizer, update scheduler.
6. Định kỳ evaluate, lưu history và checkpoint.

In [ ]:
for step in range(1, MAX_STEPS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    step_loss = 0.0

    for _ in range(GRADIENT_ACCUMULATION_STEPS):
        try:
            inputs, targets = next(train_iterator)
        except StopIteration:
            train_iterator = iter(train_loader)
            inputs, targets = next(train_iterator)
        inputs = inputs.to(device)
        targets = targets.to(device)

        autocast_context = (
            torch.autocast(device_type='cuda', dtype=amp_dtype)
            if use_amp else nullcontext()
        )
        with autocast_context:
            logits = model(inputs, use_cache=False)
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))

        step_loss += float(loss.detach())
        scaled_loss = loss / GRADIENT_ACCUMULATION_STEPS
        if use_scaler:
            scaler.scale(scaled_loss).backward()
        else:
            scaled_loss.backward()
        tokens_seen += inputs.numel()

    if use_scaler:
        scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
    if use_scaler:
        scaler.step(optimizer)
        scaler.update()
    else:
        optimizer.step()
    scheduler.step()

    should_eval = step == 1 or step % EVAL_EVERY_STEPS == 0 or step == MAX_STEPS
    if should_eval:
        train_stats = evaluate_loader(train_loader, EVAL_BATCHES_DURING_TRAINING)
        validation_stats = evaluate_loader(validation_loader, EVAL_BATCHES_DURING_TRAINING)
        record = {
            'step': step,
            'tokens_seen': tokens_seen,
            'train_loss': train_stats['loss'],
            'validation_loss': validation_stats['loss'],
            'validation_perplexity': math.exp(min(float(validation_stats['loss']), 20.0)),
            'learning_rate': scheduler.get_last_lr()[0],
            'micro_train_loss': step_loss / GRADIENT_ACCUMULATION_STEPS,
            'estimated_training_flops': tokens_seen * flops.training_flops_per_token,
            'elapsed_seconds': time.perf_counter() - start_time,
        }
        history.append(record)
        print(json.dumps(record))
        if float(validation_stats['loss']) < best_validation_loss:
            best_validation_loss = float(validation_stats['loss'])
            torch.save(model.state_dict(), RUN_DIR / 'best_model.pt')

    if step % SAVE_EVERY_STEPS == 0 or step == MAX_STEPS:
        save_notebook_checkpoint(RUN_DIR / f'checkpoint_step_{step}.pt', step)

save_notebook_checkpoint(RUN_DIR / 'checkpoint.pt', MAX_STEPS)

## 9. Final evaluation trên toàn bộ held-out data

Đây mới là số dùng cho report cuối. Không dùng training loss để kết luận architecture tốt hơn; dùng validation/test loss trên cùng fixed token streams.

In [ ]:
final_validation = evaluate_loader(validation_loader, max_batches=None)
final_test = evaluate_loader(test_loader, max_batches=None)
final_metrics = {
    'architecture': ARCHITECTURE,
    'parameters': parameter_count(model),
    'active_parameters': active_parameter_count(model, ARCHITECTURE),
    'validation_loss': final_validation['loss'],
    'validation_perplexity': math.exp(min(float(final_validation['loss']), 20.0)),
    'test_loss': final_test['loss'],
    'test_perplexity': math.exp(min(float(final_test['loss']), 20.0)),
    'validation_tokens': final_validation['tokens'],
    'test_tokens': final_test['tokens'],
    'training_flops': tokens_seen * flops.training_flops_per_token,
    'kv_cache_bytes_at_context': kv.total_bytes,
}
print(json.dumps(final_metrics, indent=2))
(RUN_DIR / 'final_metrics.json').write_text(json.dumps(final_metrics, indent=2), encoding='utf-8')

In [ ]:
steps = [row['step'] for row in history]
train_curve = [row['train_loss'] for row in history]
validation_curve = [row['validation_loss'] for row in history]
tokens_curve = [row['tokens_seen'] for row in history]

plt.figure(figsize=(9, 5))
plt.plot(tokens_curve, train_curve, label='train loss')
plt.plot(tokens_curve, validation_curve, label='validation loss')
plt.xlabel('tokens seen')
plt.ylabel('cross-entropy loss')
plt.title(f'{ARCHITECTURE}: loss vs tokens')
plt.grid(True)
plt.legend()
plt.show()

## 10. Generate text và đo accumulated KV cache

KV cache không dùng trong pretraining. Nó được dùng khi generate autoregressively để không tính lại K/V của toàn bộ prefix ở mỗi token.

In [ ]:
prompt_text = 'The small model learns'
prompt_ids = torch.tensor([tokenizer.encode(prompt_text)], dtype=torch.long, device=device)
NEW_TOKENS = 64

def decode(ids):
    return tokenizer.decode(ids[0].detach().cpu().tolist())

for use_cache in (True, False):
    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats(device)
    synchronize(device)
    start = time.perf_counter()
    generated = run_generation(model, prompt_ids, NEW_TOKENS, use_cache)
    synchronize(device)
    elapsed = time.perf_counter() - start
    cache_stats = collect_kv_cache_stats(model) if use_cache else {'bytes': 0, 'tokens': 0, 'bytes_per_token': 0.0}
    print(json.dumps({
        'use_cache': use_cache,
        'seconds': elapsed,
        'tokens_per_second': NEW_TOKENS / elapsed,
        'actual_kv_cache': cache_stats,
        'peak_vram_bytes': torch.cuda.max_memory_allocated(device) if device.type == 'cuda' else 0,
    }, indent=2))
    if use_cache:
        cached_output = generated
    else:
        uncached_output = generated

print('Cached/uncached equal:', torch.equal(cached_output, uncached_output))
print(decode(cached_output))

## 11. Optional: xem profile các architecture trước khi train nhiều model

Cell này chỉ build model và tính profile, chưa train thêm. Với MoE, hãy nhìn cả total parameters lẫn active parameters/FLOPs.

In [ ]:
profiles = []
for name in ('mha', 'gqa', 'mla', 'moe', 'v4'):
    cfg = {**base_model_config, **architecture_overrides.get(name, {})}
    candidate = build_model(name, cfg).to(device)
    candidate_flops = estimate_flops(candidate, name)
    candidate_kv = estimate_kv_cache(candidate, name, CONTEXT_LENGTH)
    profiles.append({
        'architecture': name,
        'total_parameters': parameter_count(candidate),
        'active_parameters': active_parameter_count(candidate, name),
        'forward_flops_per_token': candidate_flops.forward_flops_per_token,
        'kv_cache_bytes_per_token': candidate_kv.bytes_per_token,
    })
    del candidate
print(json.dumps(profiles, indent=2))

## Kết luận cần ghi vào report

Mỗi run nên lưu ít nhất:

```text
architecture
total parameters
active parameters
training tokens
validation/test loss và perplexity
training FLOPs
tokens/sec
peak VRAM
accumulated KV-cache bytes
```

Biểu đồ quan trọng nhất cho architecture research là `validation loss vs tokens` hoặc `validation loss vs training FLOPs`, không chỉ loss cuối cùng.